# OmniVoice Studio — Kaggle Dual-T4 + AI-native Server

Optimized for Kaggle sessions exposing **2× Tesla T4 (~15 GiB each), ~30 GiB RAM, and local SSD**.

```text
cuda:0  → OmniVoice TTS
cuda:1  → Whisper ASR verification
CPU/RAM → preprocessing, API/MCP, Gradio, file I/O
SSD     → /kaggle/working/OmniVoiceStudio + runtime-local caches
```


In [ ]:
# Resolve immutable source first. Cached revisions are never selected implicitly.
import json
import os
import re
import urllib.request

PACKAGE_REF = os.environ.get("OMNIVOICE_PACKAGE_REF", "").strip().lower()
if PACKAGE_REF:
    if re.fullmatch(r"[0-9a-f]{40}", PACKAGE_REF) is None:
        raise RuntimeError("OMNIVOICE_PACKAGE_REF must be an exact 40-character commit SHA.")
else:
    try:
        with urllib.request.urlopen(
            "https://api.github.com/repos/binhminhanh1235/OmniVoice/branches/master",
            timeout=20,
        ) as response:
            PACKAGE_REF = str(json.load(response)["commit"]["sha"]).lower()
    except Exception as exc:
        raise RuntimeError(
            "Cannot resolve current OmniVoice master exactly. Enable Internet for the small "
            "revision lookup or set OMNIVOICE_PACKAGE_REF to a verified 40-character commit SHA. "
            "A cached last_package_ref is intentionally never reused automatically."
        ) from exc
    if re.fullmatch(r"[0-9a-f]{40}", PACKAGE_REF) is None:
        raise RuntimeError("GitHub returned a non-immutable package revision.")

BOOTSTRAP_URL = (
    "https://raw.githubusercontent.com/binhminhanh1235/OmniVoice/"
    f"{PACKAGE_REF}/notebooks/hosted_runtime_bootstrap.py"
)
try:
    with urllib.request.urlopen(BOOTSTRAP_URL, timeout=30) as response:
        bootstrap_source = response.read().decode("utf-8")
except Exception as exc:
    raise RuntimeError(
        f"Cannot load the hosted-runtime bootstrap from exact revision {PACKAGE_REF}."
    ) from exc
exec(compile(bootstrap_source, BOOTSTRAP_URL, "exec"), globals(), globals())
globals().update(bootstrap_hosted_runtime(PACKAGE_REF))
del bootstrap_source

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import importlib
import torch
import omnivoice.hardware_quality as hardware_quality
from omnivoice.runtime_workspace import detect_runtime_workspace, ensure_runtime_workspace

hardware_quality = importlib.reload(hardware_quality)
runtime = ensure_runtime_workspace(detect_runtime_workspace())
if runtime.environment != "kaggle":
    raise RuntimeError(f"Expected Kaggle runtime, detected: {runtime.environment}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle Notebook settings.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GiB")

hardware = hardware_quality.detect_hardware(device_index=0)
TTS_DEVICE = "cuda:0"
ASR_DEVICE = "cuda:1" if GPU_COUNT >= 2 else hardware.recommended_asr_device
if GPU_COUNT >= 2 and hardware.recommended_asr_device != "cuda:1":
    print("WARNING: detector recommendation differs from dual-GPU runtime; forcing ASR to cuda:1.")

quality_store = hardware_quality.HardwareQualitySettingsStore(WORKSPACE)
if not quality_store.path.exists():
    quality_store.set_default(hardware.recommended_preset)
current_preset = quality_store.load().default_preset

print("Runtime:", runtime.summary())
print("Hardware:", hardware.summary())
for note in hardware.notes:
    print("-", note)
print("OmniVoice device:", TTS_DEVICE)
print("Whisper ASR device:", ASR_DEVICE)
print("Whisper ASR model:", ASR_MODEL, "@", ASR_MODEL_REVISION)
print("Workspace quality preset:", current_preset)
print("Workspace:", WORKSPACE)
usage = shutil.disk_usage("/kaggle/working")
print(f"Local SSD free: {usage.free / 1024**3:.1f} GiB")
print("Startup evidence:", STARTUP_CACHE_EVIDENCE)


## Why this mapping?

OmniVoice stays entirely on `cuda:0` instead of being sharded across both T4s. `cuda:1` becomes a dedicated Whisper accelerator for chunk verification and optional word timestamps. On a single-GPU session, ASR falls back to the detector recommendation.


## Optional: stable hostname + private access

For ChatGPT / Claude Code / other agents, create one remotely-managed Cloudflare Tunnel to `http://localhost:8000`. Create Kaggle Secrets `CLOUDFLARE_TUNNEL_TOKEN`, `OMNIVOICE_API_TOKEN`, `OMNIVOICE_UI_USERNAME`, and `OMNIVOICE_UI_PASSWORD`.


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = True

if USE_STABLE_TUNNEL:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    required = {
        "CLOUDFLARE_TUNNEL_TOKEN": secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN"),
        "OMNIVOICE_API_TOKEN": secrets.get_secret("OMNIVOICE_API_TOKEN"),
        "OMNIVOICE_UI_USERNAME": secrets.get_secret("OMNIVOICE_UI_USERNAME"),
        "OMNIVOICE_UI_PASSWORD": secrets.get_secret("OMNIVOICE_UI_PASSWORD"),
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Kaggle Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required, secrets
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /kaggle/working/cloudflared
    !/kaggle/working/cloudflared --version


## Launch OmniVoice Studio

With the stable tunnel enabled, one hostname exposes `/ui`, `/api/v1`, `/mcp`, and `/health`. If disabled, the notebook falls back to the temporary Gradio share URL.


In [ ]:
try:
    if USE_STABLE_TUNNEL:
        !omnivoice-studio serve \
          --model k2-fsa/OmniVoice \
          --device {TTS_DEVICE} \
          --workspace {WORKSPACE} \
          --asr-model {ASR_MODEL} \
          --asr-device {ASR_DEVICE} \
          --host 0.0.0.0 \
          --port 8000 \
          --tunnel \
          --cloudflared /kaggle/working/cloudflared \
          --public-url {PUBLIC_URL}
    else:
        print("Stable tunnel disabled. Using temporary Gradio share URL.")
        !omnivoice-project-studio \
          --model k2-fsa/OmniVoice \
          --device {TTS_DEVICE} \
          --workspace {WORKSPACE} \
          --asr-model {ASR_MODEL} \
          --asr-device {ASR_DEVICE} \
          --share
finally:
    persist_runtime_cache(CACHE_PREPARATION)
    write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)
    print("Startup/model cache export refreshed:", CACHE_EXPORT_BASE)


## Persistence and cache reuse

Active projects, voices, jobs, checkpoints and WAV files remain under `/kaggle/working/OmniVoiceStudio`, so they are still ephemeral unless exported or synced separately.

Startup resources are written to `/kaggle/working/OmniVoiceStartupCache`. Save/version that directory as a Kaggle Dataset named `omnivoice-startup-cache`, then attach it at `/kaggle/input/omnivoice-startup-cache` on the next run.

For measurement, record `startup-cache-evidence.json` from one cold run and one warm run. A valid warm run must report both `resource_fast_path: true` and `wheel_fast_path: true`.
